# 02 — Scoring and the Simulator-Leakage Gate

**Milestone**: M2 — Fraud Scoring and the Simulator-Leakage Gate  
**Purpose**: Train the bounded candidate set on real PaySim data, run the simulator-leakage gate, calibrate the primary, and record the evidence.  
**Scope**: The leakage gate is the *progression criterion* (FR-4, FR-26), not an after-the-fact report. A model that does not pass is ineligible regardless of headline metrics.

This notebook is descriptive/reporting. The training logic lives in `tfm.ml.train` and `evaluation/`; the same pipeline is exercised by the M2 unit tests on synthetic data.

## Setup

Set `PAYSIM_PATH` to your PaySim CSV. The pipeline reads governance/model parameters from `config/model.yaml` (split boundaries, seed, the quarantined balance-artifact features, calibration policy, and the leakage-gate decision-support defaults).

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path('..').resolve()))
sys.path.insert(0, str((Path('..') / 'src').resolve()))

import pandas as pd

from tfm.config.settings import Settings, load_model_config
from tfm.data.features import build_features
from tfm.data.ingest import load_paysim_csv

ROOT = Path('..').resolve()
PAYSIM_PATH = Path('../data/PS_20174392719_1491204439457_log.csv')
if not PAYSIM_PATH.exists():
    raise FileNotFoundError(
        f'PaySim CSV not found at {PAYSIM_PATH}. Download from Kaggle or update PAYSIM_PATH.'
    )

settings = Settings(config_dir=str(ROOT / 'config'))
model_config = load_model_config(settings)

raw = load_paysim_csv(PAYSIM_PATH)
features = build_features(raw)
print(f'Loaded and featurised {len(features):,} transactions.')
print('Split boundaries:', model_config.split.train_end_step, '/', model_config.split.val_end_step)
print('Balance-artifact features (quarantined):', model_config.balance_artifact_features)

## 1. The Bounded Candidate Set (FR-3, DF-1)

Three candidates: the interpretable HistGradientBoosting primary, the LightGBM kitchen-sink comparator, and the logistic-regression floor. The primary and floor train on the **behavioural substrate** (`PRIMARY_FEATURE_COLUMNS`) — the canonical features minus the balance artifacts quarantined after the M2 leakage FAIL (IMP-011); the comparator uses the full canonical set plus destination-balance signals. Each owns its preprocessing; the canonical feature dataset is never mutated (IMP-006).

In [ ]:
from tfm.data.features import COMPARATOR_FEATURE_COLUMNS, PRIMARY_FEATURE_COLUMNS
from tfm.ml.candidates import build_candidates

# The interpretable primary (and the logistic floor) train on the behavioural
# substrate only — the canonical features minus the quarantined balance artifacts
# (IMP-011). The kitchen-sink comparator uses the full canonical set plus the
# destination-balance signals (COMPARATOR_FEATURE_COLUMNS).
candidates = build_candidates(
    PRIMARY_FEATURE_COLUMNS, COMPARATOR_FEATURE_COLUMNS, model_config.seed
)
for cand in candidates:
    role = 'PRIMARY' if cand.is_primary else 'comparator/floor'
    print(f'{cand.name:24s} [{role}]  kind={cand.kind:9s}  n_features={len(cand.feature_columns)}')

## 2. Train + Gate + Calibrate

`run_training` splits out-of-time, fits and calibrates each candidate, evaluates on the OOT test split, and runs the leakage gate for the two DF-1 candidates. The test split is touched only for final evaluation and the gate.

In [ ]:
from tfm.ml.train import run_training

outcome = run_training(features, model_config)
report = outcome.report
print('Model version:', report.model_version_id)
print('Selected candidate eligible (passed leakage gate):', outcome.eligible)

## 3. DF-1 — Interpretable vs Kitchen-Sink (FR-5, §11.1)

Both candidates report the full metric set — PR-AUC, precision, recall, Brier — and their leakage verdict, so the interpretability comparison is aligned with the full evaluation protocol, not raw discrimination alone.

In [ ]:
df1 = report.df1
comparison = pd.DataFrame([df1['interpretable'], df1['kitchen_sink']])
print(comparison.to_string(index=False))
print()
print('PR-AUC delta (kitchen-sink - interpretable):',
      df1['pr_auc_delta_kitchen_minus_interpretable'])
print('Interpretability decision:', df1['interpretability_decision'])

## 4. Calibration (FR-23)

Probability calibration in the reliability sense, fitted on the validation split (isotonic or Platt, chosen with the small-fold guard). A predicted 0.7 should correspond to roughly 70% observed fraud.

In [ ]:
cal = report.calibration
print('Calibration method:', cal.method)
print(f'Brier before: {cal.brier_before:.5f}   Brier after: {cal.brier_after:.5f}')
print()
reliability = pd.DataFrame([b.model_dump() for b in cal.bins])
print(reliability.to_string(index=False))

## 5. Leakage-Gate Evidence (FR-26, §9)

The verdict is evidence-based (IMP-007). The evidence: the full model vs the ablated (balance-artifact-free) model, the permutation-importance inspection, and the behavioural performance that survives ablation. The configured numbers are decision-support defaults, not the definition of the decision.

In [ ]:
verdict = report.selected_leakage_verdict
ev = verdict.evidence

print('Full model    PR-AUC:', round(ev.full_metrics.pr_auc, 4))
print('Ablated model PR-AUC:', round(ev.ablated_metrics.pr_auc, 4))
print('Ablation delta (PR-AUC drop):', round(ev.pr_auc_delta, 4))
print('Behavioural PR-AUC surviving ablation:', round(ev.remaining_behavioural_pr_auc, 4))
print('Balance-artifact share of total importance:',
      f'{ev.balance_artifact_importance_share:.1%}')
print()
print('Top permutation importances:')
importances = pd.DataFrame([fi.model_dump() for fi in ev.top_importances])
print(importances.to_string(index=False))

## Leakage Verdict Summary

This is the engineering record of the gate and the artifact for the final presentation. It names the primary model, the ablation model, the verdict, and the evidence supporting it.

In [ ]:
primary_report = report.candidate_reports[0]

print('=' * 68)
print('LEAKAGE VERDICT SUMMARY')
print('=' * 68)
print(f'Model version      : {report.model_version_id}')
print(f'Primary model      : {primary_report.name} (interpretable HistGradientBoosting)')
print(f'Ablation model     : {primary_report.name}_ablated '
      f'(balance-artifact features removed)')
print(f'Verdict            : {verdict.verdict.upper()}  '
      f'(eligible for online path: {outcome.eligible})')
print('-' * 68)
print('Supporting evidence:')
print(f'  Full-model PR-AUC (OOT test)        : {ev.full_metrics.pr_auc:.4f}')
print(f'  Ablated-model PR-AUC (OOT test)     : {ev.ablated_metrics.pr_auc:.4f}')
print(f'  Ablation delta (PR-AUC drop)        : {ev.pr_auc_delta:.4f}')
print(f'  Behavioural PR-AUC after ablation   : {ev.remaining_behavioural_pr_auc:.4f}')
print(f'  Balance-artifact importance share   : {ev.balance_artifact_importance_share:.1%}')
print(f'  Applied decision-support defaults   : {verdict.applied_defaults}')
print('-' * 68)
print('Rationale:')
print(verdict.rationale)
print('=' * 68)
print('NOTE: all metrics are measured on synthetic PaySim data (an optimistic')
print('upper bound on real-world performance, per the data strategy, §6/§9).')